# CS781: one BiLSTM prediction for each of Phases 2-7, in one notebook (about 3-4 hours on a T4)

The Phase 1 BiLSTM (560 steps) is fine-tuned **once per phase** under a time cap, and each phase writes a schema-checked `prediction.json` to `/kaggle/working/predictions/phase_N/`.
Phases 3-7 reuse the cells of the notebooks you already ran on Kaggle (patched only to run the BiLSTM experiment once); Phase 2 is a new BiLSTM tagger followed by your counsel/JUDGE rule layer.

**Attach:** the dataset `cs781-phase1-v3-bilstm-recurrent` (final_model.pt + legal_bpe_32k_mask.json). **Secrets:** `HF_TOKEN` (and `KAGGLE_USERNAME` / `KAGGLE_KEY` if a child's credentials check asks for them). **Settings:** Internet ON, GPU T4 x1.

**Honest status:** the Phase 2 child was run on CPU end to end with a tiny model and synthetic data. Phases 3-7 are your own cells with small patches; they were checked for syntax and patch targets only, and **have not been run** with the BiLSTM. Phase 6 uses the existing S7 BiLSTM sequence-to-sequence model (not the encoder + GPT-2 decoder variant). Run order and times are estimates until the first real run.

In [ ]:
# ================= CONFIG =================
RUN_PHASES = [2, 3, 4, 7]                  # Phases 5 and 6 already produced BiLSTM predictions in the first run; add them back to rerun
CAP_MIN = {2: 25, 3: 0, 4: 25, 5: 25, 6: 30, 7: 20}   # BiLSTM training minutes per phase (Phase 3 has no minute cap: it trains 3 epochs on about 6-10k sentences)
CHILD_TIMEOUT_MIN = {2: 70, 3: 60, 4: 90, 5: 90, 6: 100, 7: 70}   # hard stop per phase, so one stuck phase cannot eat the session
SELFTEST = False
print("planned training minutes:", {p: CAP_MIN[p] for p in RUN_PHASES}, "| hard stops:", {p: CHILD_TIMEOUT_MIN[p] for p in RUN_PHASES})

In [ ]:
# ================= SUPPORT FILES (written once) =================
import os, json, gzip, base64, textwrap
ROOT = os.environ.get('CS781_ROOT', '/kaggle/working'); os.makedirs(ROOT, exist_ok=True)
ARCH_LOADERS_SRC = '"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.\n   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)\n   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""\nimport torch, torch.nn as nn\nfrom types import SimpleNamespace\nfrom transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration\n\nclass BiLSTMLM(nn.Module):\n    def __init__(self, vocab, emb=512, hidden=1024, layers=3):\n        super().__init__()\n        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers\n        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)\n        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)\n        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))\n    def hidden_states(self, input_ids, attention_mask=None):\n        x = input_ids; B, T = x.shape\n        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)\n        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)\n        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))\n        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))\n        return torch.cat([hf, hb], dim=-1)\n\nclass Backbone(nn.Module):\n    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""\n    def __init__(self, family, inner, embed):\n        super().__init__(); self.family, self.inner, self._embed = family, inner, embed\n    def get_input_embeddings(self): return self._embed\n    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):\n        f = self.family\n        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state\n        else:\n            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"\n            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)\n            else: raise ValueError(f)\n        return SimpleNamespace(last_hidden_state=h)\n\nCAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}\n# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT\'s q,k,v are ONE fused linear, \'Wqkv\' (rows: q | k | v).\nADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}\n\ndef build_full(mc):\n    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]\n    if f == "gpt2":\n        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),\n                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))\n    if f == "modernbert":\n        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],\n                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,\n                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))\n    if f == "t5":\n        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],\n                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))\n    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])\n    raise ValueError(f)\n\ndef load_backbone(path, device="cpu"):\n    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)\n    f = mc["family"]\n    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte\n    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings\n    elif f == "t5": inner, embed = full.encoder, full.shared\n    elif f == "bilstm": inner, embed = full, full.emb\n    else: raise ValueError(f)\n    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f\n'
RULES_SRC = 'import re, random\n\nA_, R_ = \'A.COUNSEL\', \'R.COUNSEL\'\n\nchosen = \'R\'   # counsel default policy; select_defaults() re-chooses it on fold 1\n\nENT = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT","AUTH","COURT","STAT","PREC","DATE","CASENO"]\n\nTYPES = ["O"] + ENT\n\nWEAK = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]\n\nHON = re.compile(r"\\b(?:Mr|Ms|Mrs|Shri|Sri|Smt|Dr)\\b\\.?[ \\t]*(?=[A-Z])")\n\nHON_RE  = re.compile(r"\\b(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\b\\.?[ \\t]*(?=[A-Z])")\n\nNAME_TOK = re.compile(r"[A-Z][A-Za-z\\.\']*")\n\nWS = re.compile(r"[ \\t]+")\n\nSIDE_A = re.compile(r"petitioner|appellant|applicant|plaintiff|complainant|prosecution", re.I)\n\nSIDE_R = re.compile(r"respondent|defendant|accused", re.I)\n\ndef rule_spans(text):\n    out = []\n    for m in HON_RE.finditer(text):\n        pos = m.end()\n        if text[pos:pos + 7] == "Justice":          # judges\' honorific, not counsel\n            continue\n        p, name_end = pos, None\n        while True:\n            t = NAME_TOK.match(text, p)\n            if not t: break\n            name_end = t.end()\n            w = WS.match(text, name_end)\n            if not w or not text[w.end():w.end() + 1].isupper(): break\n            p = w.end()\n        if name_end is None: continue\n        e = name_end\n        while e > pos and text[e - 1] in ".,;:\'": e -= 1\n        if e > pos: out.append((pos, e))\n    return out\n\nHON_BEFORE = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\.?[ \\t]*$")\n\nPARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"\n\nPARTY_R = r"(?:respondents?|defendants?|accused|non-\\s*applicants?|opposite\\s+part(?:y|ies))"\n\nFOR = r"(?:for|on\\s+behalf\\s+of|appearing\\s+for|representing)"\n\nMODS = r"(?:(?:learned|first|second|third|\\d+\\w*|all|other)\\s+)*"\n\nCUE_A = re.compile(rf"\\b{FOR}\\s+(?:the\\s+)?{MODS}{PARTY_A}", re.I)\n\nCUE_R = re.compile(rf"\\b{FOR}\\s+(?:the\\s+)?{MODS}{PARTY_R}", re.I)\n\nHDR_A = re.compile(rf"{PARTY_A}s?[\\s\\.\\-:]*(?:through|by)\\s*[:\\-]?\\s*$", re.I)\n\nHDR_R = re.compile(rf"{PARTY_R}s?[\\s\\.\\-:]*(?:through|by)\\s*[:\\-]?\\s*$", re.I)\n\nHON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\.?[ \\t]*$")\n\ndef _cues(text, lo, hi):\n    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]\n    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]\n    return out\n\ndef assign_side(text, s, e, order, prev, W_L=300, W_R=200):\n    for step in order:\n        if step == "right":\n            c = _cues(text, e, min(len(text), e + W_R))\n            if c: return min(c)[2], "right"\n        elif step == "left":\n            c = _cues(text, max(0, s - W_L), s)\n            c = [x for x in c if x[1] <= s]\n            if c: return max(c, key=lambda x: x[1])[2], "left"\n        elif step == "hdr":\n            tail = HON_TAIL.sub("", text[max(0, s - 80):s])\n            if HDR_A.search(tail): return A_, "hdr"\n            if HDR_R.search(tail): return R_, "hdr"\n        elif step == "inherit":\n            if prev and s - prev[0] <= 200 and prev[2] != "default":\n                return prev[1], "inherit"\n    return R_, "default"\n\nVARIANTS = {"V1 left-for": ["left"], "V2 right-for": ["right"], "V3 right>left": ["right", "left"],\n            "V4 left>right": ["left", "right"], "V5 hdr>right>left>inh": ["hdr", "right", "left", "inherit"],\n            "V6 right>hdr>left>inh": ["right", "hdr", "left", "inherit"]}\n\nORDER = ["hdr", "right", "left", "inherit"]\n\ndef counsel_spans(text, policy):\n    out, prev = [], None\n    for s, e in sorted(rule_spans(text)):\n        lab, src = assign_side(text, s, e, ORDER, prev)\n        prev = (e, lab, src)\n        if src == "default":\n            if policy == "drop": continue\n            if policy == "A": lab = A_\n        out.append((s, e, lab))\n    return out\n\ndef counsel_f1(texts, gold, policy):\n    G = {A_: set(), R_: set()}; P = {A_: set(), R_: set()}\n    for d, ents in gold.items():\n        for x in ents:\n            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))\n    for d, text in texts.items():\n        for s, e, lab in counsel_spans(text, policy): P[lab].add((d, s, e))\n    f = {}\n    for lab in (A_, R_):\n        tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)\n        f[lab] = 2 * p * r / (p + r) if p + r else 0.0\n    return f\n\nLOOK = ["APP", "RESP", "WIT", "JUDGE"]\n\nTOK = r"(?:[A-Z]\\.(?:\\s?[A-Z]\\.)*|[A-Z][a-z\']+|[A-Z]{2,})"\n\nTOK_RE = re.compile(TOK)\n\nNAME = rf"{TOK}(?:[ \\t]+{TOK}){{0,3}}"\n\nSTOP = {"JUDGMENT","JUDGEMENT","ORDER","BENCH","CORAM","BEFORE","PRESENT","HON\'BLE","MR","MRS","MS",\n        "JUSTICE","THE","AND","OF","DATE","NEW","DELHI","BY","DELIVERED"}\n\ndef back_name(text, end, max_toks=4):\n    ws = max(0, end - 80)\n    toks = [(m.start() + ws, m.end() + ws) for m in TOK_RE.finditer(text[ws:end])]\n    run = []\n    for a, b in reversed(toks):\n        nxt = run[0][0] if run else end\n        if text[b:nxt].strip(" \\t") != "": break\n        if text[a:b].upper().strip(".") in STOP: break\n        run.insert(0, (a, b))\n        if len(run) == max_toks: break\n    return (run[0][0], run[-1][1]) if run else None\n\ndef trim(text, s, e):\n    while s < e and (text[s].isspace() or text[s] in ".,;:"): s += 1\n    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"): e -= 1\n    return (s, e) if e > s else None\n\nA_RE = re.compile(rf"\\bJustice[ \\t]+({NAME})")\n\nB_RE = re.compile(r",[ \\t]*J{1,2}\\b\\.?")\n\nC_RE = re.compile(r"\\bJJ?\\.?[ \\t]*\\(([^()\\n]{3,45})\\)")\n\nD_RE = re.compile(r"(?:BENCH\\s*:|\\(J\\)|\\(J\\.\\))\\s*([A-Z][A-Za-z\']+,[ \\t]*[A-Z](?:[ \\t]*\\.?[ \\t]*[A-Z])*\\.?)[ \\t]*\\(J\\)")\n\nCAP   = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ\'’\\-]*"\n\nINIT  = r"[A-Z]\\.(?:[ \\t]*[A-Z]\\.)*"\n\nWORD  = rf"(?:{INIT}|{CAP})"\n\nNAME2 = rf"{WORD}(?:[ \\t]+{WORD}){{0,3}}"\n\nTOK_RE = re.compile(WORD)\n\nAJ_RE = re.compile(rf"(?i:\\bjustice)[ \\t]+({NAME2})")\n\nB_RE  = re.compile(r",[ \\t]*J{1,2}\\b\\.?")\n\nB2_RE = re.compile(r"[ \\t]J{1,2}\\.?(?=[ \\t]*(?:in\\b|speaking|writing|observed|held\\b|was\\b|said\\b|while\\b|who\\b|delivered|dissent|agreed|also\\b|noted|stated))")\n\nC_RE  = re.compile(r"\\bJJ?\\.?[ \\t]*\\(([^()\\n]{3,45})\\)")\n\nD_RE  = re.compile(r"(?:BENCH\\s*:|\\(J\\)|\\(J\\.\\))\\s*([A-Z][A-Za-z\']+,[ \\t]*[A-Z](?:[ \\t]*\\.?[ \\t]*[A-Z])*\\.?)[ \\t]*\\(J\\)")\n\nE_RE  = re.compile(r"\\bJJ?\\.?[ \\t]*\\[[ \\t]*([A-Za-zÀ-ÿ\\.\'’ \\t\\-]{3,45}?)[ \\t]*\\]")\n\nF_RE  = re.compile(r"BENCH[ \\t]*:[ \\t]*([^\\n]{3,120}?)(?=[ \\t]*(?:JUDG(?:E)?MENT|CITATION|ACT\\b|\\n|$))")\n\nNAMES = ["A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)", "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"]\n\ndef judge_patterns2(text):\n    out = {n: set() for n in NAMES}\n    def add(n, s, e):\n        sp = trim(text, s, e)\n        if sp: out[n].add(sp)\n    for m in AJ_RE.finditer(text): add(NAMES[0], m.start(1), m.end(1))\n    for rx, n in [(B_RE, NAMES[1]), (B2_RE, NAMES[2])]:\n        for m in rx.finditer(text):\n            bn = back_name(text, m.start())\n            if bn: add(n, *bn)\n    for rx, n in [(C_RE, NAMES[3]), (E_RE, NAMES[5])]:\n        for m in rx.finditer(text):\n            inner = m.group(1).strip()\n            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} and re.fullmatch(r"[A-Za-zÀ-ÿ\\.\'’, \\-]+", inner):\n                add(n, m.start(1), m.end(1))\n    for m in D_RE.finditer(text): add(NAMES[4], m.start(1), m.end(1))\n    for m in F_RE.finditer(text):\n        seg, base, pos = m.group(1), m.start(1), 0\n        for part in re.split(r"[ \\t]*(?:&|\\bAND\\b)[ \\t]*", seg):\n            k = seg.find(part, pos)\n            if k < 0: continue\n            pos = k + len(part)\n            part2 = re.sub(r"[ \\t]*\\((?:C?J|J\\.?)\\)\\s*$", "", part)\n            sp = trim(text, base + k, base + k + len(part2))\n            if sp and re.fullmatch(NAME2, text[sp[0]:sp[1]]): out[NAMES[6]].add(sp)\n    return out\n\ndef resolve(spans):\n    out = []\n    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):\n        if out and s < out[-1][1]: continue\n        out.append((s, e))\n    return out\n\ndef judge_spans_doc(text, keep):\n    allsp = set()\n    for n, sp in judge_patterns2(text).items():\n        if n in keep: allsp |= sp\n    return resolve(allsp)\n\ndef merge_all(texts, champ_filtered, keep):\n    out = {}\n    for d, text in texts.items():\n        rules = list(counsel_spans(text, chosen)) + [(s, e, "JUDGE") for s, e in judge_spans_doc(text, keep)]\n        kept = [k for k in champ_filtered[d] if k[2] not in (A_, R_, "JUDGE")]\n        kept = [k for k in kept if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]\n        out[d] = sorted(set(kept + rules))\n    return out'
CHILDREN_B64 = '...'
open(ROOT + '/arch_loaders.py', 'w', encoding='utf-8').write(ARCH_LOADERS_SRC)
open(ROOT + '/p2_rules_core.py', 'w', encoding='utf-8').write(RULES_SRC)
CHILDREN = json.loads(gzip.decompress(base64.b64decode(CHILDREN_B64)).decode('utf-8'))
print({k: len(v) for k, v in CHILDREN.items()}, 'cells per phase')

In [ ]:
# ================= RUNNER: one phase = one subprocess (its own Python process, so variables of different phases never collide) =================
import os, sys, json, time, glob, shutil, subprocess, traceback, zipfile
ROOT = os.environ.get("CS781_ROOT", "/kaggle/working")
PRED_DIR = os.path.join(ROOT, "predictions"); os.makedirs(PRED_DIR, exist_ok=True)
NB_T0 = time.time()
RERUN = False                      # True = run a phase again even if its prediction already exists in predictions/phase_N/

def child_script(phase, cap):
    cells = list(CHILDREN[str(phase)]); cells[0] = cells[0].replace("{cap}", str(cap))
    if SELFTEST and phase == 2: cells[1] = cells[1].replace("SELFTEST = False          # the runner patches this", "SELFTEST = True")
    d = os.path.join(ROOT, "children"); os.makedirs(d, exist_ok=True)
    json.dump(cells, open(os.path.join(d, f"phase{phase}_cells.json"), "w", encoding="utf-8"))
    script = ("import json, sys, traceback\nG = {'__name__': '__main__'}\ncells = json.load(open(%r, encoding='utf-8'))\n"
              "for i, src in enumerate(cells):\n    print('\\n===== phase %d, cell %%d =====' %% i, flush=True)\n"
              "    try: exec(compile(src, 'phase%d_cell%%d' %% i, 'exec'), G)\n"
              "    except BaseException as e:\n        if isinstance(e, KeyboardInterrupt): raise\n        print('CELL FAILED:', type(e).__name__, e, flush=True); traceback.print_exc()\n") % (os.path.join(d, f"phase{phase}_cells.json"), phase, phase)
    p = os.path.join(d, f"phase{phase}_run.py"); open(p, "w", encoding="utf-8").write(script); return p

def find_output(phase):
    cands = {2: ["phase2_bilstm/prediction.json"], 3: ["phase3_rr/prediction.json"], 4: ["cjpe/submissions/P5_bilstm_partial*.json"], 5: ["lsi/submissions/E4_bilstm_partial*.json"],
             6: ["p6/submissions/S7_bilstm_partial*.json"], 7: ["pcr_bilstm/prediction_bilstm_only.json", "pcr_bilstm/prediction.json", "**/prediction_bilstm_only.json"]}[phase]
    for pat in cands:
        hits = [h for h in glob.glob(os.path.join(ROOT, pat), recursive=True) if os.path.isfile(h) and PRED_DIR not in h and "/children/" not in h.replace("\\", "/")]
        if hits: return sorted(hits, key=os.path.getmtime)[-1]
    return None

def validate(phase, path):
    o = json.load(open(path, encoding="utf-8"))
    if phase == 2:
        assert isinstance(o, dict) and o, "expected {doc_id: [[start, end, label], ...]}"
        for d, sp in o.items():
            for x in sp: assert len(x) == 3 and isinstance(x[0], int) and isinstance(x[1], int) and x[0] < x[1] and isinstance(x[2], str), f"bad span {x} in {d}"
        return f"{len(o)} docs, {sum(map(len, o.values()))} spans"
    if phase == 3:
        assert isinstance(o, dict) and set(o) >= {"CL", "IT"}, "expected keys CL and IT (CHECK this against the official Phase 3 schema)"
        return "keys " + ",".join(o) + " (schema is the pipeline's 'keyed' mode: verify against the spec)"
    if phase == 4:
        assert isinstance(o, list) and all(set(x) >= {"id", "pred"} and x["pred"] in (0, 1) for x in o), "expected [{id, pred in {0,1}}]"
        return f"{len(o)} cases, {sum(x['pred'] for x in o) / len(o):.1%} predicted 1"
    if phase == 5:
        assert isinstance(o, list) and all(set(x) >= {"id", "pred"} and isinstance(x["pred"], list) and len(set(x["pred"])) == len(x["pred"]) and all(isinstance(v, int) for v in x["pred"]) for x in o), "expected [{id, pred: [unique ints]}]"
        return f"{len(o)} cases, {sum(len(x['pred']) for x in o) / len(o):.2f} statutes/case"
    if phase == 6:
        assert isinstance(o, list) and all(set(x) >= {"id", "summary"} and isinstance(x["summary"], str) for x in o), "expected [{id, summary}]"
        return f"{len(o)} summaries, {sum(len(x['summary'].split()) for x in o) / max(len(o), 1):.0f} words on average"
    if phase == 7:
        assert isinstance(o, dict) and all(isinstance(v, list) and len(v) == 7 and len(set(v)) == 7 for v in o.values()), "expected {query_id: [7 unique candidate ids]}"
        return f"{len(o)} queries x 7"

def run_phase(phase):
    dst = os.path.join(PRED_DIR, f"phase_{phase}"); os.makedirs(dst, exist_ok=True)
    if os.path.exists(os.path.join(dst, "prediction.json")) and not RERUN:
        return dict(phase=phase, status="already done (kept)", minutes=0.0, check=validate(phase, os.path.join(dst, "prediction.json")))
    cap = CAP_MIN[phase]; script = child_script(phase, cap); log = os.path.join(dst, "log.txt"); t0 = time.time()
    env = dict(os.environ)
    if SELFTEST: env.update(P2_WORK=os.path.join(ROOT, "phase2_bilstm"), P2_RULES_PATH=os.path.join(ROOT, "p2_rules_core.py"))
    status = "ok"
    with open(log, "w", encoding="utf-8") as lf:
        pr = subprocess.Popen([sys.executable, "-u", script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", env=env, cwd=ROOT)
        try:
            for line in pr.stdout:
                lf.write(line); print(line, end="")
                if time.time() - t0 > CHILD_TIMEOUT_MIN[phase] * 60: pr.kill(); status = f"STOPPED at the {CHILD_TIMEOUT_MIN[phase]}-minute hard limit"; break
        finally:
            if pr.poll() is None: pr.kill()
    out = find_output(phase); check = ""
    if out:
        try: check = validate(phase, out); shutil.copy2(out, os.path.join(dst, "prediction.json"))
        except Exception as e: status += f" | OUTPUT INVALID: {e}"; shutil.copy2(out, os.path.join(dst, "prediction_INVALID.json"))
    else: status += " | no prediction file was produced (read log.txt)"
    for extra in ("results.json", "prediction_model_only.json"):
        for h in glob.glob(os.path.join(ROOT, "**", extra), recursive=True):
            if "phase2_bilstm" in h.replace("\\", "/"): shutil.copy2(h, os.path.join(dst, extra))
    return dict(phase=phase, status=status, minutes=round((time.time() - t0) / 60, 1), check=check, source=out)

RESULTS = []
for ph in RUN_PHASES:
    print(f"\n######## PHASE {ph}  (notebook time so far {(time.time() - NB_T0) / 60:.0f} min) ########", flush=True)
    try: RESULTS.append(run_phase(ph))
    except Exception as e: traceback.print_exc(); RESULTS.append(dict(phase=ph, status=f"RUNNER ERROR: {e}", minutes=0.0, check=""))
    json.dump(RESULTS, open(os.path.join(PRED_DIR, "summary.json"), "w"), indent=1, default=str)

# ================= SUMMARY AND ZIP =================
print("\n================ SUMMARY ================")
for r in RESULTS: print(f"Phase {r['phase']}: {r['status']:<60} {r['minutes']:>6} min | {r['check']}")
print(f"total notebook time: {(time.time() - NB_T0) / 60:.0f} min")
zp = os.path.join(ROOT, "bilstm_predictions.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for h in glob.glob(os.path.join(PRED_DIR, "**", "*"), recursive=True):
        if os.path.isfile(h): z.write(h, os.path.relpath(h, PRED_DIR))
print("wrote", zp)
